# Демо 1. Класичне донавчання: зміна голови в моделі BERT

У цьому ноутбуці донавчаємо модель **BERT** для задачі **визначення парафраз**: модель отримує два речення й визначає, чи передають вони той самий зміст.

План:
1. Завантажити датасет MRPC і токенізувати пари речень.
2. Завантажити BERT із **новою головою**-класифікатором.
3. **Повне донавчання** з `Trainer`: тренуються всі ваги (кістяк і голова).
4. Перевірити модель на власних прикладах.

Приклад за мотивами розділу 3 курсу [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/chapter3/1).
Для запуску в Colab оберіть середовище з GPU: *Runtime → Change runtime type → T4 GPU*.

In [1]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [2]:
# Фіксуємо seed, щоб результати повторювалися між запусками
set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Пристрій: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("Включіть GPU, бо набридне чекати ;)")

Пристрій: cuda
GPU: Tesla T4


## 1. Дані: MRPC

**MRPC** (Microsoft Research Paraphrase Corpus) — пари речень з новинних статей.
Мітка `1` означає, що речення є парафразами, мітка `0` — що ні.

In [3]:
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

mrpc/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  649kB            

mrpc/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 75.7kB            

mrpc/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

mrpc/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  308kB            

mrpc/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [4]:
# Подивимося на один приклад
print("Приклад нееквівалентних речень:")
example = raw_datasets["train"][3]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

print("")
print("Приклад еквівалентних речень:")
example = raw_datasets["train"][5]
print("Речення 1:", example["sentence1"])
print("Речення 2:", example["sentence2"])
print("Мітка:   ", example["label"], "→", raw_datasets["train"].features["label"].names[example["label"]])

Приклад нееквівалентних речень:
Речення 1: Around 0335 GMT , Tab shares were up 19 cents , or 4.4 % , at A $ 4.56 , having earlier set a record high of A $ 4.57 .
Речення 2: Tab shares jumped 20 cents , or 4.6 % , to set a record closing high at A $ 4.57 .
Мітка:    0 → not_equivalent

Приклад еквівалентних речень:
Речення 1: Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .
Речення 2: With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .
Мітка:    1 → equivalent


### Знайомимось з BERT BASE UNCASED

Для цього прочитайте інформацію на [картці моделі](https://huggingface.co/google-bert/bert-base-uncased)

In [5]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model='bert-base-uncased')
unmasker("The man worked as a [MASK].")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[{'score': 0.09747552871704102,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man worked as a carpenter.'},
 {'score': 0.052383244037628174,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man worked as a waiter.'},
 {'score': 0.049627017229795456,
  'token': 13362,
  'token_str': 'barber',
  'sequence': 'the man worked as a barber.'},
 {'score': 0.037886038422584534,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man worked as a mechanic.'},
 {'score': 0.037680864334106445,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man worked as a salesman.'}]

In [6]:
print("Не по темі, але погляньте, яка \"несправедливість\"...")
unmasker("The woman worked as a [MASK].")

Не по темі, але погляньте, яка "несправедливість"...


[{'score': 0.21981368958950043,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the woman worked as a nurse.'},
 {'score': 0.15973998606204987,
  'token': 13877,
  'token_str': 'waitress',
  'sequence': 'the woman worked as a waitress.'},
 {'score': 0.11547233909368515,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the woman worked as a maid.'},
 {'score': 0.03796861320734024,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the woman worked as a prostitute.'},
 {'score': 0.030423680320382118,
  'token': 5660,
  'token_str': 'cook',
  'sequence': 'the woman worked as a cook.'}]

### Токенізація пари речень

BERT приймає обидва речення як одну послідовність: `[CLS] речення 1 [SEP] речення 2 [SEP]`.
Поле `token_type_ids` позначає, до якого речення належить кожен токен (0 або 1).

Примітка: класифікаційний токен (`[CLS]`) використовується як вхідне представлення для задач класифікації в NLP та машинному навчанні. Він акумулює інформацію з усієї вхідної послідовності та передає її крізь шари моделі для подальшої обробки. Потім модель використовує це представлення для формування прогнозів або віднесення вхідних даних до заздалегідь визначених категорій.

In [7]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
encoded = tokenizer(example["sentence1"], example["sentence2"])

print("")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[:20])
print(encoded["token_type_ids"][:20])
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"])[-20:])
print(encoded["token_type_ids"][-20:])


['[CLS]', 'revenue', 'in', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.']
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[',', 'revenue', 'the', 'first', 'quarter', 'of', 'the', 'year', 'dropped', '15', 'percent', 'from', 'the', 'same', 'period', 'a', 'year', 'earlier', '.', '[SEP]']
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [8]:
def tokenize_function(batch):
    # truncation=True обрізає надто довгі пари до максимальної довжини моделі
    return tokenizer(batch["sentence1"], batch["sentence2"], truncation=True)


tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Доповнення (padding) робимо динамічно для кожного батча — до найдовшого прикладу в ньому
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized_datasets

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

## 2. Модель: кістяк + нова голова

`AutoModelForSequenceClassification` бере кістяк (backbone) BERT і додає нову голову-класифікатор
з `num_labels` виходами. Зверніть увагу на звіт під час завантаження:
- ваги `cls.*` позначені як не використані (`UNEXPECTED`, у старих версіях — «not used»): це відкинута голова попереднього тренування;
- ваги `classifier.*` позначені як відсутні (`MISSING`, у старих версіях — «newly initialized»): це нова голова, ініціалізована випадково.

In [17]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

print(model.classifier)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Linear(in_features=768, out_features=2, bias=True)


In [20]:
def count_params(module, trainable_only=False):
    return sum(p.numel() for p in module.parameters() if p.requires_grad or not trainable_only)


print(f"Кістяк (model.bert):   {count_params(model.bert):>12,} параметрів")
print(f"Голова (model.classifier): {count_params(model.classifier):>8,} параметрів")

Кістяк (model.bert):    109,482,240 параметрів
Голова (model.classifier):    1,538 параметрів


## 3. Повне донавчання з `Trainer`

`Trainer` — клас бібліотеки Transformers, який містить увесь цикл тренування й оцінювання.
Метрики рахуємо самі у функції `compute_metrics`.

Оновлюються **всі** ваги: і кістяк, і нова голова.
Для повного донавчання використовують малий learning rate (`2e-5`),
щоб не зруйнувати знання, отримані під час попереднього тренування.

In [21]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }


training_args = TrainingArguments(
    output_dir="bert-mrpc",
    num_train_epochs=3,             # зазвичай вистачає 1-3 епохи
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",          # оцінюємо на валідації після кожної епохи
    save_strategy="no",             # чекпоінти не зберігаємо
    logging_steps=50,
    fp16=torch.cuda.is_available(), # змішана точність пришвидшує тренування на GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [22]:
print(f"Параметрів, що тренуються: {count_params(model, trainable_only=True):,}")
trainer.train()
trainer.evaluate()

Параметрів, що тренуються: 109,483,778


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.522325,0.458917,0.808824,0.866894
2,0.383182,0.388196,0.843137,0.890411
3,0.219031,0.439131,0.845588,0.892675


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.219031,0.439131,3,0.845588,0.892675


{'eval_loss': 0.4391307830810547,
 'eval_accuracy': 0.8455882352941176,
 'eval_f1': 0.8926746166950597}

## 4. Перевірка на власних прикладах

In [23]:
def predict(model, sentence1, sentence2):
    model.eval()
    inputs = tokenizer(sentence1, sentence2, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = model(**inputs).logits.softmax(dim=-1)[0]
    return f"парафраз: {probs[1]:.1%}"


pairs = [
    ("The company reported a profit of $5 million.", "The firm earned five million dollars in profit."),
    ("The company reported a profit of $5 million.", "The weather was sunny all week."),
    ("She bought a new car yesterday.", "Yesterday she purchased a new vehicle."),
    # Власні перевірки
    ("The cat is sleeping on the sofa.", "A cat is resting on the couch."),
    ("The train arrived at the station.", "The train departed from the station."),
    ("He enjoys playing football.", "He likes playing soccer."),
    ("The restaurant is closed today.", "The restaurant is open today.")
]
for s1, s2 in pairs:
    print(f"{s1}\n{s2}\n→ {predict(model, s1, s2)}\n")

The company reported a profit of $5 million.
The firm earned five million dollars in profit.
→ парафраз: 94.4%

The company reported a profit of $5 million.
The weather was sunny all week.
→ парафраз: 9.3%

She bought a new car yesterday.
Yesterday she purchased a new vehicle.
→ парафраз: 99.4%

The cat is sleeping on the sofa.
A cat is resting on the couch.
→ парафраз: 99.1%

The train arrived at the station.
The train departed from the station.
→ парафраз: 96.9%

He enjoys playing football.
He likes playing soccer.
→ парафраз: 92.2%

The restaurant is closed today.
The restaurant is open today.
→ парафраз: 99.4%



#0.1. Теоретичні питання

###1 Як довго виконувалися клітинки ноутбуку 01_bert_head? Чи були проблеми з запуском, які?
Загалом швидко. Одні виконувалися від 1 до 5 секунд. Найдовше виконувалася остання клітинка з 3 розділу, 58 секунд. Проблем з запуском ніяких не було.

###2 Як Ви розумієте, що таке токен [CLS]? Яким чином він виконує свою функцію?
Токен `[CLS]` — це спеціальний токен, який розміщується на початку послідовності тексту. Він використовується для узагальнення інформації з усієї послідовності, щоб надалі модель могла краще зрозуміти її загальний зміст і використати його для класифікації.

###3 Яке пряме застосування базової моделі bert-base-uncased?

bert-base-uncased — це попередньо навчена модель для роботи з англійським текстом, яка використовується для розуміння контексту слів і речень. Її можна застосовувати як основу для різних завдань обробки природної мови, наприклад класифікації тексту, визначення його тональності та аналізу зв'язку між реченнями.

###4 Як Ви вважаєте, чи варто боротися (і якщо так, то яким чином) з проявами сексизму LLM-моделей?

На мою думку, з проявами сексизму LLM-моделей варто боротися. При цьому сама модель не винна в таких проявах, оскільки її відповіді базуються на даних, на яких вона навчалася. Тому проблема часто полягає саме в навчальних даних, які можуть містити стереотипи та упередження.

Для боротьби з цим можна використовувати додаткові правила поведінки моделі, наприклад задати їй роль ввічливого та неупередженого помічника. Також варто перевіряти навчальні дані та позначати приклади, які містять стереотипи або некоректні висловлювання, щоб модель не сприймала їх як норму спілкування.

###5 Для чого в секції "Токенізація пари речень" виводяться числа 0 і 1?

Числа `0` і `1` позначають, до якого речення належить кожен токен. `0` відповідає першому реченню, а `1` — другому. Це допомагає моделі BERT розрізняти два речення, навіть коли вони обробляються як одна спільна послідовність.

###6 Що позначають числа 768 і 2, що виводяться після завантаження AutoModelForSequenceClassification.from_pretrained()?

Число `768` позначає розмір вхідного набору числових значень, які передаються від BERT до класифікаційного шару. Число `2` позначає кількість класів, між якими модель повинна виконати класифікацію. Тобто модель використовує 768 вхідних значень для отримання результату, який відповідає одному з двох класів.

###7 Те що в кістяку (backbone) набагато більше параметрів ніж в голові, це добре чи погано в нашому випадку?

На мою думку, це добре і цілком логічно в нашому випадку. Кістяк моделі містить основну частину параметрів та вже має знання, отримані під час попереднього навчання. Класифікаційна голова значно менша, оскільки її завдання — адаптувати ці знання для вирішення конкретної задачі класифікації. Тобто нам не потрібно створювати та навчати модель з нуля, а достатньо налаштувати вже готову модель під наше завдання.

###8 Що саме вимірюється за допомогою метрик accuracy & F1 в 01_bert_head?

Метрика Accuracy показує відсоток правильно класифікованих прикладів від їх загальної кількості. Метрика F1 оцінює якість класифікації, враховуючи точність і повноту відповідей моделі. У нашому випадку Accuracy становить приблизно 86%, а F1 — 90,2%. Це означає, що модель досить добре справляється із завданням класифікації.

###9 Збільшіть кількість епох тренування в 01_bert_head, покажіть (скопіювавши результати) що відбувається і чому.

При збільшенні кількості епох з 3 до 9 спостерігається перенавчання моделі (overfitting). Це видно за результатами: Training Loss поступово зменшується майже до нуля, тоді як Validation Loss зростає. Також показники Accuracy та F1 дещо погіршилися.

Це означає, що модель почала надто добре запам'ятовувати тренувальні дані, але втратила здатність так само добре працювати з новими прикладами. Отже, у нашому випадку збільшення кількості епох не покращило результат, а навпаки призвело до перенавчання.
Фіксуємо результати:
`Параметрів, що тренуються: 109,483,778

 [2070/2070 03:18, Epoch 9/9]

Epoch	Training Loss	Validation Loss	Accuracy	F1

1	0.077380	0.725152	0.840686	0.883721

2	0.069839	0.798198	0.845588	0.891566

3	0.065374	0.881925	0.838235	0.881720

4	0.057866	1.005992	0.828431	0.882943

5	0.017942	1.189090	0.828431	0.883721

6	0.023554	1.140910	0.833333	0.885522

7	0.003670	1.259206	0.825980	0.882255

8	0.003044	1.096811	0.840686	0.887348

9	0.000169	1.146602	0.830882	0.883249

 [7/7 00:00]

Training Loss	Validation Loss	Epoch	Accuracy	F1

0.000169	1.146602	9	0.830882	0.883249
{'eval_loss': 1.1466023921966553,
 'eval_accuracy': 0.8308823529411765,
 'eval_f1': 0.883248730964467}`


###10 Додайте ще 4 перевірки на власних даних в пункті 4 в 01_bert_head.
Під час перевірки на власних прикладах модель правильно визначила більшість пар речень, які мали однаковий зміст. Проте виявилися помилки: речення з протилежним значенням, наприклад «поїзд прибув» і «поїзд відправився», а також «ресторан зачинений» і «ресторан відчинений», були визначені як парафрази.

Причиною таких результатів є перенавчання моделі, яке ми зробили під час виконанян завдання 9.

Після повторного запуску тестування результати дещо змінилися. Модель почала видавати різні відсоткові значення замість лише 0% або 100%. Більшість пар речень була визначена правильно, однак залишилися помилки.

Зокрема, модель помилково визначила речення «The train arrived at the station» та «The train departed from the station» як парафрази з результатом 96,9%. Аналогічна ситуація виникла з реченнями «The restaurant is closed today» та «The restaurant is open today», результат склав 99,4%.


